In [11]:
import json, re, time, random
from datetime import datetime, timezone
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# =========================
# Config
# =========================
LIST_URL = "https://www.kca.go.kr/odr/cm/in/exmplPgItem.do"
OUT_JSONL = "kca_00000006_full.jsonl"
ERROR_JSONL = "kca_00000006_errors.jsonl"

START_PAGE = 1
END_PAGE = 71     # 총 705건 / 10개(페이지당) = 71페이지 (너가 확인한 값 기준)

TIMEOUT = 20
MIN_SLEEP = 1.3
MAX_SLEEP = 2.8

# 500 뜨면 쿨다운(서버가 세션/속도 제한 걸 때)
COOLDOWN_SEC = 20

# =========================
# Utils
# =========================
def now_iso():
    return datetime.now(timezone.utc).isoformat()

def human_sleep(a=MIN_SLEEP, b=MAX_SLEEP):
    time.sleep(random.uniform(a, b))

def append_jsonl(path: str, obj: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def load_seen_ids(path: str) -> set:
    seen = set()
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    o = json.loads(line)
                    _id = o.get("id")
                    if _id:
                        seen.add(str(_id))
                except:
                    pass
    except FileNotFoundError:
        pass
    return seen

def make_id(brdId: str, seq: str) -> str:
    return f"board:{brdId}:{seq}"

def make_public_url(brdId: str, seq: str) -> str:
    # ✅ 너가 원한 "붙일 url"
    return f"https://www.kca.go.kr/odr/cm/cm/boardsDtl.do?brdId={brdId}&seq={seq}&dataStts=Y"

def parse_fn_view_bbd(onclick: str):
    # fn_view_bbd("1003634144", "00000006") or single quotes
    m = re.search(r"fn_view_bbd\(\s*['\"](\d+)['\"]\s*,\s*['\"](\d+)['\"]\s*\)", onclick or "")
    if not m:
        return None, None
    return m.group(1), m.group(2)

def normalize_text(t: str) -> str:
    if not t:
        return ""
    t = t.replace("\u00a0", " ")
    t = re.sub(r"\n{3,}", "\n\n", t)
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

def safe_text(el) -> str:
    return normalize_text(el.get_text("\n", strip=True)) if el else ""

def parse_detail_html(html: str):
    soup = BeautifulSoup(html, "html.parser")

    title = safe_text(soup.select_one("div.board_v_tit h4"))

    updated_at = ""
    views = ""
    vtbl = soup.select_one("table.v_tbl")
    if vtbl:
        for tr in vtbl.select("tr"):
            ths = tr.select("th")
            tds = tr.select("td")
            if not ths or not tds:
                continue

            pairs = []
            if len(ths) == len(tds):
                for i in range(len(ths)):
                    pairs.append((ths[i].get_text(strip=True), safe_text(tds[i])))
            else:
                # web/mobile 섞여 있을 수 있어 전체를 훑게끔
                for th in ths:
                    td = th.find_next("td")
                    if td:
                        pairs.append((th.get_text(strip=True), safe_text(td)))

            for k, v in pairs:
                if k == "수정일" and v and not updated_at:
                    updated_at = v
                if k == "조회수" and v and not views:
                    views = v

    question = ""
    answer = ""
    q_tr = soup.select_one("tr.qna_q")
    a_tr = soup.select_one("tr.qna_a")
    if q_tr:
        q_span = q_tr.select_one("span span") or q_tr.select_one("span")
        question = safe_text(q_span) if q_span else safe_text(q_tr)
        question = question.replace("질문", "").strip()
    if a_tr:
        a_span = a_tr.select_one("span span") or a_tr.select_one("span")
        answer = safe_text(a_span) if a_span else safe_text(a_tr)
        answer = answer.replace("답변", "").strip()

    # ✅ UI 텍스트(board_v_btn)는 soup가 알아서 안 넣지만,
    # 혹시라도 내용에 섞일 여지 없게 content는 우리가 만든 필드만으로 구성
    parts = []
    parts.append("[문서유형] 게시판 문서")
    parts.append("[문서타입] consumer_relief_case")
    if title: parts.append(f"[제목] {title}")
    if updated_at: parts.append(f"[수정일] {updated_at}")
    if views: parts.append(f"[조회수] {views}")
    if question: parts.append(f"[질문]\n{question}")
    if answer: parts.append(f"[답변]\n{answer}")
    content = "\n\n".join(parts).strip()

    return title, updated_at, views, question, answer, content

def is_500_block(html: str) -> bool:
    return ("500 ERROR" in html) or ("서비스 이용이 원활하지 않습니다" in html)

# =========================
# Selenium main
# =========================
# Chrome options
opts = webdriver.ChromeOptions()
opts.add_argument("--start-maximized")
# opts.add_argument("--headless=new")  # 처음엔 끄는 걸 추천

driver = webdriver.Chrome(options=opts)
wait = WebDriverWait(driver, TIMEOUT)

seen = load_seen_ids(OUT_JSONL)
print("seen already:", len(seen))

try:
    driver.get(LIST_URL)
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.tbl_col table tbody tr")))

    saved = 0
    skipped = 0
    failed = 0

    for pg in range(START_PAGE, END_PAGE + 1):
        print(f"\n[list] page={pg}")

        # 페이지 이동 (정상 플로우)
        if pg > 1:
            driver.execute_script("fn_link_page(arguments[0])", str(pg))
            human_sleep(1.0, 2.0)
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.tbl_col table tbody tr")))

        rows = driver.find_elements(By.CSS_SELECTOR, "div.tbl_col table tbody tr")
        items = []

        # 목록에서 메타 수집
        for r in rows:
            try:
                a = r.find_element(By.CSS_SELECTOR, "td.al_l a")
                onclick = a.get_attribute("onclick")
                seq, brdId = parse_fn_view_bbd(onclick)
                if not seq or not brdId:
                    continue

                tds = r.find_elements(By.CSS_SELECTOR, "td")
                no = tds[0].text.strip() if len(tds) > 0 else ""
                title_list = a.text.strip()
                source_list = tds[2].text.strip() if len(tds) > 2 else ""
                views_list = tds[3].text.strip() if len(tds) > 3 else ""

                items.append({
                    "seq": seq,
                    "brdId": brdId,
                    "no": no,
                    "title_list": title_list,
                    "source_list": source_list,
                    "views_list": views_list,
                    "onclick": onclick,
                })
            except:
                continue

        print(f"[list] items={len(items)}")

        # 상세 순차 수집
        for meta in items:
            seq = meta["seq"]
            brdId = meta["brdId"]
            doc_id = make_id(brdId, seq)

            if doc_id in seen:
                skipped += 1
                continue

            human_sleep()

            try:
                # ✅ 정상 클릭 플로우로 상세 진입
                driver.execute_script("fn_view_bbd(arguments[0], arguments[1])", seq, brdId)
                wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.board_view")))

                html = driver.page_source
                if is_500_block(html):
                    # 쿨다운 후 목록으로 복귀 및 같은 페이지로 재진입
                    append_jsonl(ERROR_JSONL, {
                        "id": doc_id,
                        "seq": seq,
                        "brdId": brdId,
                        "page": pg,
                        "error": "blocked_500",
                        "at": now_iso(),
                    })
                    failed += 1
                    time.sleep(COOLDOWN_SEC)

                    driver.get(LIST_URL)
                    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.tbl_col table tbody tr")))
                    if pg > 1:
                        driver.execute_script("fn_link_page(arguments[0])", str(pg))
                        human_sleep(1.0, 2.0)
                    continue

                title, updated_at, views, question, answer, content = parse_detail_html(html)

                # 최소 검증
                if not (title or question or answer):
                    raise RuntimeError("empty parsed fields")

                # ✅ 최종 doc (id 다음에 url 오도록)
                doc = {
                    "id": doc_id,
                    "url": make_public_url(brdId, seq),
                    "title": title,
                    "updated_at": updated_at,
                    "views": views,
                    "question": question,
                    "answer": answer,
                    "content": content,
                    "collected_at": now_iso(),
                    "metadata": {
                        "site": "kca.go.kr",
                        "doc_type": "consumer_relief_case",
                        "board_code": brdId,
                        "post_id": seq,
                        "source_list": meta.get("source_list", ""),
                        "views_list": meta.get("views_list", ""),
                    },
                    "list_meta": {
                        "list_page": pg,
                        "no": meta.get("no", ""),
                        "title_list": meta.get("title_list", ""),
                        "source_list": meta.get("source_list", ""),
                        "views_list": meta.get("views_list", ""),
                    }
                }

                append_jsonl(OUT_JSONL, doc)
                seen.add(doc_id)
                saved += 1

                # ✅ 목록으로 복귀: "목록보기" 버튼 클릭(정상 플로우)
                try:
                    btn = driver.find_element(By.CSS_SELECTOR, "div.board_v_btn a.btn_default.empt_blue")
                    btn.click()
                    human_sleep(1.0, 2.0)
                    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.tbl_col table tbody tr")))
                except:
                    # 버튼 실패하면 목록으로 재진입 + 같은 페이지 복귀
                    driver.get(LIST_URL)
                    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.tbl_col table tbody tr")))
                    if pg > 1:
                        driver.execute_script("fn_link_page(arguments[0])", str(pg))
                        human_sleep(1.0, 2.0)

            except Exception as e:
                failed += 1
                append_jsonl(ERROR_JSONL, {
                    "id": doc_id,
                    "seq": seq,
                    "brdId": brdId,
                    "page": pg,
                    "error": repr(e),
                    "at": now_iso(),
                })

                # 실패 시에도 목록으로 복귀 + 같은 페이지
                driver.get(LIST_URL)
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.tbl_col table tbody tr")))
                if pg > 1:
                    driver.execute_script("fn_link_page(arguments[0])", str(pg))
                    human_sleep(1.0, 2.0)

        print(f"✅ checkpoint page={pg} saved={saved} skipped={skipped} failed={failed}")

    print("\n==== DONE ====")
    print(f"saved={saved} skipped={skipped} failed={failed}")

finally:
    driver.quit()


seen already: 0

[list] page=1
[list] items=10
✅ checkpoint page=1 saved=10 skipped=0 failed=0

[list] page=2
[list] items=10
✅ checkpoint page=2 saved=20 skipped=0 failed=0

[list] page=3
[list] items=10
✅ checkpoint page=3 saved=30 skipped=0 failed=0

[list] page=4
[list] items=10
✅ checkpoint page=4 saved=40 skipped=0 failed=0

[list] page=5
[list] items=10
✅ checkpoint page=5 saved=50 skipped=0 failed=0

[list] page=6
[list] items=10
✅ checkpoint page=6 saved=60 skipped=0 failed=0

[list] page=7
[list] items=10
✅ checkpoint page=7 saved=70 skipped=0 failed=0

[list] page=8
[list] items=10
✅ checkpoint page=8 saved=80 skipped=0 failed=0

[list] page=9
[list] items=10
✅ checkpoint page=9 saved=90 skipped=0 failed=0

[list] page=10
[list] items=10
✅ checkpoint page=10 saved=100 skipped=0 failed=0

[list] page=11
[list] items=10
✅ checkpoint page=11 saved=110 skipped=0 failed=0

[list] page=12
[list] items=10
✅ checkpoint page=12 saved=120 skipped=0 failed=0

[list] page=13
[list] item